In [1]:
import pandas as pd
import numpy as np
import glob
import os
import shutil
import matplotlib.pyplot as plt

import mne
import pyprep 

from mne.preprocessing import read_ica
from mne_icalabel import label_components
from mne_bids.write import _write_json

from mne_bids import BIDSPath, read_raw_bids, get_entity_vals, write_raw_bids

import meegkit.dss

%matplotlib widget

In [ ]:
# Set the base directory and the task and run details
base_dir = '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline'
task = 'restEyesClosed'
run = '01'

# Get a list of all subjects (assumes the subject directories are named 'sub-XXXXXX')
subjects = sorted([d for d in os.listdir(base_dir) if d.startswith('sub-')])

# Loop through each subject and session
for subject in subjects:
    subject_dir = os.path.join(base_dir, subject)
    
    # Get a list of sessions for each subject (assumes session directories are named 'ses-XXXXXX')
    sessions = [d for d in os.listdir(subject_dir) if d.startswith('ses-')]
    
    for session in sessions:
        # Construct the full path to the FIF file
        eeg_file = os.path.join(
            subject_dir,
            session,
            'eeg',
            f'{subject}_{session}_task-{task}_run-{run}_proc-filt_raw.fif'
        )
        # eeg_file2 = os.path.join(
        #     subject_dir,
        #     session,
        #     'eeg',
        #     f'{subject}_{session}_task-{task}_run-{run}_proc-filt_raw2.fif'
        # )


        # Check if the file exists before loading
        if os.path.exists(eeg_file):
            print(f'Loading file: {eeg_file}')

            raw = mne.io.read_raw_fif(eeg_file, preload=True)

            # process the 'raw' object
            sfreq = raw.info['sfreq']  # Sampling frequency from the MNE Raw object
            line_freq = 50  # Set to 50 Hz or 60 Hz depending on your region
            
            # Extract the data from the Raw object
            data = raw.get_data()  # shape will be (n_channels, n_samples)

            # meegkit expects data in the form (n_samples, n_chans, n_trials), so we need to transpose and reshape
            # Assuming no trials (continuous data), we can add a third dimension for trials (1 trial)
            data_for_meegkit = data.T[:, :, np.newaxis]  # shape (n_samples, n_chans, n_trials)

            # Apply the iterative DSS line noise removal
            # cleaned_data, iterations = meegkit.dss.dss_line_iter(
            #                                                         data=data_for_meegkit,
            #                                                         fline=line_freq,
            #                                                         sfreq=sfreq,
            #                                                         win_sz = 5,
            #                                                         spot_sz = 2,
            #                                                         #nfft=400,
            #                                                         n_iter_max=100,
            #                                                     )
            cleaned_data, iterations = meegkit.dss.dss_line(
                                                                    data_for_meegkit,
                                                                    fline=line_freq,
                                                                    sfreq=sfreq,
                                                                    nremove=6,
                                                                    nfft=512,
                                                                    #nkeep=None
                                                                )


            # Remove the extra trial dimension and transpose the data back to MNE's format
            cleaned_data_mne = cleaned_data[:, :, 0].T  # shape back to (n_channels, n_samples)

            # Create a new MNE Raw object with the cleaned data
            cleaned_raw = mne.io.RawArray(cleaned_data_mne, raw.info)

            # Save the manipulated raw object back to the same file (overwrite)
            cleaned_raw.save(eeg_file, overwrite=True)
            print(f'Saved powerline cleaned raw file: {eeg_file}')

        #else:
        #    print(f'File not found: {eeg_file}')

File not found: /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20181115/eeg/sub-NZ000052_ses-20181115_task-restEyesClosed_run-01_proc-filt_raw.fif
File not found: /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20191022/eeg/sub-NZ000052_ses-20191022_task-restEyesClosed_run-01_proc-filt_raw.fif
File not found: /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20170911/eeg/sub-NZ000052_ses-20170911_task-restEyesClosed_run-01_proc-filt_raw.fif
File not found: /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20170927/eeg/sub-NZ000052_ses-20170927_task-restEyesClosed_run-01_proc-filt_raw.fif
File not found: /media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline/sub-NZ000052/ses-20200811/eeg/sub-NZ000052_ses-20200811_task-restEyesCl

In [ ]:
import os
import mne
from mne.report import Report

# Set the base directory and the task and run details
base_dir = '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline'
task = 'restEyesClosed'
run = '01'

# Initialize the MNE report
report = Report(title="EEG PSD Report", verbose=True)

# Get a list of all subjects (assumes the subject directories are named 'sub-XXXXXX')
subjects = sorted([d for d in os.listdir(base_dir) if d.startswith('sub-')])

# Loop through each subject and session
for subject in subjects:
    subject_dir = os.path.join(base_dir, subject)
    
    # Get a list of sessions for each subject (assumes session directories are named 'ses-XXXXXX')
    sessions = [d for d in os.listdir(subject_dir) if d.startswith('ses-')]
    
    for session in sessions:
        # Construct the full path to the FIF file
        eeg_file = os.path.join(
            subject_dir,
            session,
            'eeg',
            f'{subject}_{session}_task-{task}_run-{run}_proc-filt_raw.fif'
        )
        
        # Check if the file exists before loading
        if os.path.exists(eeg_file):
            print(f'Processing file: {eeg_file}')

            # Load the raw EEG data
            raw = mne.io.read_raw_fif(eeg_file, preload=True)
            
            # Compute the PSD for the raw data
            psd_fig = raw.compute_psd(fmax=120).plot(show=False)  # Show is disabled
            
            # Add the PSD figure to the MNE report
            caption = f"{subject} - {session}: PSD plot"
            report.add_figure(fig=psd_fig, title=caption, caption=caption, tags=(subject, session))
        #else:
        #    print(f'File not found: {eeg_file}')

# Save the report to the base directory
report_file = os.path.join(base_dir, 'EEG_PSD_Report.html')
report.save(report_file, overwrite=True)
print(f'Report saved to: {report_file}')


NameError: name 'os' is not defined